# HFP Demo — Cihaz-içi Hafıza (geri getirme süper-gücü)

**Dürüst beklenti.** Bu model Qwen2.5-1.5B **base** (chat-tuned DEĞİL) + 6-katman
O(1) hafıza graft'ı (RESULTS §22, 1.11× PPL, 3 seed). Yani sohbet zekâsı beklemeyin
— ham tamamlama yapar. Gösterdiği şey **kanıtlanmış olan**: uzun bir metnin içine
gömülü bir detayı, minik **sabit** bir hafıza durumundan, yerelde geri getirmek.

**Ne görürsünüz:** metni akıtırsınız (istediğiniz kadar uzun) → HFP hafıza durumu
sabit ~birkaç MB kalır (bağlamla büyümez) → sonra sorarsınız → gömülü detayı
hatırlar. Bu, "cihazda, gizli, unutmayan hafıza" vizyonunun elle tutulur hâli.

**Gerekli:** eğitilmiş 6-katman checkpoint (`hfp_graft_exp_g6*_final.pt`, Run 7/8/9).
Drive'da veya oturumda erişilebilir olmalı; kod arar.


In [ ]:
# --- 1. KURULUM ---
import os, subprocess, sys, glob
os.environ['HF_HUB_DISABLE_XET']='1'; os.environ['HF_HUB_ENABLE_HF_TRANSFER']='0'
import torch
assert torch.cuda.is_available(), 'GPU sec (Runtime>T4).'
DEV='cuda'
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else '.')
REPO = os.path.join(BASE,'HFP')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers>=4.46','accelerate','gradio'],check=True)
os.chdir(REPO); sys.path.insert(0,REPO)
CKPT_DIR=BASE
try:
    from google.colab import drive; drive.mount('/content/drive')
    CKPT_DIR='/content/drive/MyDrive/hfp_graft_ckpt'
except Exception as e: print('Drive yok:',type(e).__name__)
# HF token (gerekirse elle koy): os.environ['HF_TOKEN']='hf_...'
print('kurulum tamam')

In [ ]:
# --- 2. MODEL + GRAFT + EGITILMIS AGIRLIKLARI YUKLE ---
from transformers import AutoModelForCausalLM, AutoTokenizer
from hfp.models.grafting import (GraftConfig, graft_llama, set_graft_mode,
                                 enable_streaming, reset_streaming, HFPGraftAttention)
MODEL_ID='Qwen/Qwen2.5-1.5B'; GRAFT_LAYERS=[3,7,11,15,19,23]

def _find(pats, roots):
    for r in roots:
        if not r or not os.path.isdir(r): continue
        for p in pats:
            h=glob.glob(f'{r}/**/{p}',recursive=True)
            if h: return sorted(h)[-1]
    return None
ROOTS=['/kaggle/input',CKPT_DIR,BASE,'/content/drive/MyDrive','/content']
src=_find(['config.json'],ROOTS)
src=os.path.dirname(src) if (src and glob.glob(f'{os.path.dirname(src)}/*.safetensors')) else None
if not src:
    _tk=os.environ.get('HF_TOKEN'); assert _tk,'Model yok + HF_TOKEN yok. Ilk hucrede os.environ[\'HF_TOKEN\']=... koy.'
    from huggingface_hub import snapshot_download
    src=f'{BASE}/qwen_model'
    snapshot_download(repo_id=MODEL_ID,local_dir=src,token=_tk,
        allow_patterns=['*.json','*.txt','tokenizer.model','*.safetensors','*.safetensors.index.json'],max_workers=2)
tok=AutoTokenizer.from_pretrained(src)
model=AutoModelForCausalLM.from_pretrained(src,torch_dtype=torch.float32).to(DEV).eval()
graft_llama(model, GraftConfig(decay_mode='exp',write_rule='hybrid',key_feature_map='dpfp',rec_block=16), layers=GRAFT_LAYERS)
for m in model.modules():
    if isinstance(m,HFPGraftAttention): m.out_gain.data.fill_(0.1)

CKPT=_find(['hfp_graft_exp_g6*_final.pt','hfp_graft*g6*final*.pt','hfp_graft*g6*son*.pt'],ROOTS)
assert CKPT, ('Egitilmis 6-kat checkpoint bulunamadi (hfp_graft_exp_g6*_final.pt).\n'
              '  Run 7/8/9 ciktisi Drive/hfp_graft_ckpt altinda olmali; yoksa ana notebook\'u\n'
              '  GRAFT_N=6 ile bir kez kosup uret.')
sd=torch.load(CKPT,map_location=DEV)
missing,unexpected=model.load_state_dict(sd,strict=False)
set_graft_mode(model,'student')
print(f'Checkpoint yuklendi: {CKPT}')
print(f'  HFP param yuklendi (base zaten frozen). eslesmeyen HFP: '
      f'{len([k for k in sd if k in [n for n,_ in model.named_parameters()]])}/{len(sd)}')

In [ ]:
# --- 3. STREAMING GERI-GETIRME + GRADIO ARAYUZU ---
from transformers import DynamicCache
import gradio as gr

@torch.no_grad()
def hfp_recall(memory_text, question, max_new=48, chunk=512):
    set_graft_mode(model,'student'); model.eval()
    enable_streaming(model,True); reset_streaming(model)
    cache=DynamicCache()
    ids=tok((memory_text or '')+(question or ''), add_special_tokens=False, return_tensors='pt').input_ids.to(DEV)
    n_ctx=ids.size(1); out=None
    for s in range(0,ids.size(1),chunk):
        out=model(ids[:,s:s+chunk],past_key_values=cache,use_cache=True); cache=out.past_key_values
    gen=[]; last=out.logits[:,-1:].argmax(-1)
    for _ in range(max_new):
        gen.append(last.item())
        out=model(last,past_key_values=cache,use_cache=True); cache=out.past_key_values
        last=out.logits[:,-1:].argmax(-1)
        if last.item()==tok.eos_token_id: break
    mb=0.0
    for m in model.modules():
        if isinstance(m,HFPGraftAttention) and m._stream_state is not None:
            M,z=m._stream_state[0],m._stream_state[1]
            mb+=(M.numel()*M.element_size()+z.numel()*z.element_size())/1e6
    enable_streaming(model,False)
    ans=tok.decode(gen).strip()
    info=f'Baglam: {n_ctx:,} token  |  HFP hafiza durumu: {mb:.1f} MB (SABIT — baglam uzadikca BUYUMEZ)'
    return ans, info

# Varsayilan ornek: gomulu gizli detay (needle) — kutudan cikar cikmaz calisir
_filler=' The weather was ordinary and nothing of note happened that day.'*30
_EX_MEM=(_filler[:len(_filler)//2] +
         ' The secret passphrase is purple ladder. ' + _filler[len(_filler)//2:])
_EX_Q=' The secret passphrase is'

with gr.Blocks(title='HFP — Cihaz-ici Hafiza') as demo:
    gr.Markdown('## HFP — cihazda, gizli, unutmayan hafiza\n'
                'Uzun bir metni akit (icine bir detay goml), sonra sor. Model detayi '
                '**sabit** birkac MB hafizadan geri getirir. (Qwen2.5-1.5B base + O(1) graft; '
                'sohbet degil, HATIRLAMA gosterir.)')
    with gr.Row():
        mem=gr.Textbox(label='Hafiza (uzun metin — icine detay goml)',lines=10,value=_EX_MEM)
        with gr.Column():
            q=gr.Textbox(label='Soru / devam',value=_EX_Q)
            n=gr.Slider(8,96,48,step=8,label='uretilecek token')
            btn=gr.Button('Hatirla',variant='primary')
    out=gr.Textbox(label='Model cevabi')
    info=gr.Markdown()
    btn.click(hfp_recall,[mem,q,n],[out,info])

demo.launch(share=True)   # share=True -> herkese acik link (birine gostermek icin)